# MobileNetV2

This standalone notebook trains **MobileNetV2**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.
Enable `COPY_DATASET_TO_WORKING` in the setup cell to optionally copy the validated dataset to Kaggle working storage for faster I/O.


In [ ]:
import os
from pathlib import Path

def find_dataset_root(input_root='/kaggle/input'):
    candidates = []
    input_root = Path(input_root)
    for root, directories, _ in os.walk(input_root):
        root_path = Path(root)
        if len(root_path.relative_to(input_root).parts) > 6:
            directories[:] = []
            continue
        if {'train', 'val', 'test'}.issubset(directories):
            candidates.append(root_path)
            directories[:] = []
    if not candidates:
        raise FileNotFoundError(
            f'No dataset containing train, val, and test folders was found under {input_root}. ' 
            'Attach the leaf-disease dataset through Kaggle Add Input.'
        )
    def rank(path):
        class_count = sum(child.is_dir() for child in (path / 'train').iterdir())
        return (class_count == 72, class_count, len(path.parts))
    return str(max(candidates, key=rank))

DATASET_BASE = find_dataset_root()
print('Selected dataset root:', DATASET_BASE)
for split in ('train', 'val', 'test'):
    split_path = Path(DATASET_BASE) / split
    if split_path.is_dir():
        print(f'  OK {split}: {sum(child.is_dir() for child in split_path.iterdir())} classes')
    else:
        raise FileNotFoundError(f'Required dataset split not found: {split_path}')

In [ ]:
import os
import sys

ARCH = "mobilenetv2"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
KAGGLE_INPUT_ROOT = "/kaggle/input"
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = "/kaggle/working/leaf_dataset/processed"
OUTPUT_BASE = "/kaggle/working/leaf_models"
print("Configured:", ARCH)
print("Dataset search root:", KAGGLE_INPUT_ROOT)
print("Outputs:", OUTPUT_BASE)
print("Copy dataset to Kaggle working storage:", COPY_DATASET_TO_WORKING)


## Environment and repository setup

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
BRANCH = 'main'
repo_dir = REPO_DIR
required_paths = [
    'requirements.txt',
    'training/train.py',
    'training/evaluate.py',
    'quantization/post_training_quant.py',
    'quantization/evaluate_tflite.py',
    'quantization/qat.py',
    'training/config_mobilenetv2.yaml',
]

repo_is_complete = all(
    os.path.exists(os.path.join(repo_dir, path)) for path in required_paths
)
if not os.path.exists(repo_dir):
    clone_result = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, repo_dir],
        text=True,
        capture_output=True,
        check=False,
    )
    if clone_result.returncode != 0:
        details = '\n'.join(
            output for output in (clone_result.stdout, clone_result.stderr) if output
        ).strip()
        raise RuntimeError(
            f'Could not clone {REPO_URL!r} branch {BRANCH!r}. '
            'Enable Kaggle Internet and verify that the repository is public or '
            'configure GitHub authentication in Kaggle Secrets.\n'
            f'Git output:\n{details}'
        )
elif not repo_is_complete:
    raise RuntimeError(
        'The existing repo directory is incomplete. Restart the Kaggle session or '
        'remove only /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System, then rerun this cell.'
    )
else:
    print('Complete repository already exists:', repo_dir)

missing_paths = [
    path for path in required_paths
    if not os.path.isfile(os.path.join(repo_dir, path))
]
if missing_paths:
    raise FileNotFoundError(
        'Repository is missing required training files: ' + ', '.join(missing_paths)
    )

print('Repository branch ready:', BRANCH)
print('Repository ready:', repo_dir)

## Install project dependencies

In [ ]:
%pip install -q -r '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/requirements.txt'
%pip install -q tensorflow-model-optimization
print('Dependencies installed successfully!')

In [ ]:
import os
import shutil

os.environ["TF_USE_LEGACY_KERAS"] = "1"
print("TF_USE_LEGACY_KERAS:", os.environ["TF_USE_LEGACY_KERAS"])

## Configure paths and detect classes

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', f'config_{ARCH}.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

# Prefer the exact-deduplicated, manifest-backed benchmark when the optional
# preparation cell below has built one. A mounted dataset that already ships
# its own split_manifest.csv (for example a locally rebuilt cap-500 benchmark
# uploaded as a Kaggle dataset) is equally manifest-backed: split validation
# is instant (no image hashing) and the metrics reported by the evaluate cell
# are leakage-free.
DEDUP_BENCHMARK_DIR = '/kaggle/working/leaf_dataset/benchmark_dedup'
MANIFEST_SOURCE = None
if os.path.isfile(os.path.join(DEDUP_BENCHMARK_DIR, 'split_manifest.csv')):
    dataset_base = DEDUP_BENCHMARK_DIR
    MANIFEST_SOURCE = DEDUP_BENCHMARK_DIR
    print('Using deduplicated benchmark:', dataset_base)
elif os.path.isfile(os.path.join(dataset_base, 'split_manifest.csv')):
    MANIFEST_SOURCE = dataset_base
    print('Using manifest-backed mounted dataset:', dataset_base)

output_base = '/kaggle/working/leaf_models'
OUTPUT_BASE = output_base
ARCH_OUTPUT_DIR = os.path.join(output_base, ARCH)
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

# Split validation: enabled with a manifest (finishes in seconds, honest
# leakage-free counts); disabled on the raw mounted input, whose exact
# duplicate scan across splits takes minutes and cannot be fixed there.
cfg.setdefault('performance', {})
if MANIFEST_SOURCE:
    cfg['performance']['validate_dataset_splits'] = True
    cfg['performance']['allow_cross_split_duplicates'] = True
else:
    cfg['performance']['validate_dataset_splits'] = False
    cfg['performance'].pop('allow_cross_split_duplicates', None)

# Keep all training hyperparameters in the architecture config.
# This cell only supplies Kaggle paths and the discovered class count.
cfg['export']['save_dir'] = output_base
cfg['export']['sync_flutter_labels'] = False
cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_{}.csv'.format(output_base, ARCH)

# The next cell applies the Kaggle runtime profile to this configured file.
runtime_cfg_path = cfg_path
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Configuration updated for Kaggle')
print('Split validation: {}'.format(
    'enabled (manifest-backed benchmark)'
    if MANIFEST_SOURCE
    else 'disabled (mounted input without split_manifest.csv)'
))
print('Training schedule: {} frozen epochs + {} fine-tune epochs'.format(
    cfg['training']['unfreeze_epoch'],
    cfg['training']['epochs'] - cfg['training']['unfreeze_epoch'],
))
print('Fine-tune from layer: {}; LR: {}; clipnorm: {}'.format(
    cfg['training']['unfreeze_from_layer'],
    cfg['training']['fine_tune_learning_rate'],
    cfg['optimizer'].get('clipnorm'),
))
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Train dataset: {}'.format(cfg['dataset']['train_dir']))
print('Models save to: {}'.format(cfg['export']['save_dir']))
print('TensorBoard logs: DISABLED (saves space)')


## Optional: prepare the deduplicated benchmark (thesis pass)

In [ ]:
# OPTIONAL - run only when building a fresh benchmark on Kaggle. It rebuilds
# every split on an exact-deduplicated, manifest-backed tree (10-20 min).
# Once /kaggle/working/leaf_dataset/benchmark_dedup/split_manifest.csv exists,
# re-run the configure cell above: it auto-switches to this benchmark and
# re-enables split validation, which the manifest makes nearly instant.
#
# When is this cell needed?
# * The active dataset already ships split_manifest.csv (for example a
#   locally rebuilt cap-500 benchmark uploaded to Kaggle): nothing to do -
#   this cell exits early with a note unless you override the cap.
# * The active dataset is the OLD capped train/val/test mount (no manifest):
#   rebuilding from it cannot raise the per-class cap, so this cell refuses.
#   Attach the uncapped raw class-folder dataset (dataset/raw) instead.
# To rebuild on Kaggle explicitly, set MAX_PER_CLASS (e.g. 500) or
# FORCE_BENCHMARK_REBUILD=1 in the notebook environment and re-run.
import os
import sys
import json

benchmark_dir = '/kaggle/working/leaf_dataset/benchmark_dedup'
benchmark_manifest = os.path.join(benchmark_dir, 'split_manifest.csv')
max_per_class = int(os.environ.get('MAX_PER_CLASS', '500'))
if max_per_class < 3:
    raise ValueError('MAX_PER_CLASS must be an integer >= 3')

force_rebuild = os.environ.get(
    'FORCE_BENCHMARK_REBUILD', ''
).strip().lower() in {'1', 'true', 'yes', 'on'}

active_root = dataset_base if 'dataset_base' in globals() else DATASET_BASE
active_manifest = os.path.join(active_root, 'split_manifest.csv')
active_cap = None
active_info = os.path.join(active_root, 'dataset_info.json')
if os.path.isfile(active_info):
    try:
        with open(active_info, 'r', encoding='utf-8') as f:
            active_cap = json.load(f).get('max_per_class')
    except (OSError, ValueError):
        active_cap = None

if not force_rebuild and os.path.isfile(active_manifest):
    if os.path.isfile(benchmark_manifest):
        print('Deduplicated benchmark already present:', benchmark_dir)
        print('Active dataset is also manifest-backed - nothing to rebuild.')
    else:
        print(
            'Active dataset is already manifest-backed '
            f'(cap per class: {active_cap}). Skip building a benchmark; '
            'set MAX_PER_CLASS and FORCE_BENCHMARK_REBUILD=1 to rebuild.'
        )
else:
    def _looks_like_raw_class_tree(root):
        if not os.path.isdir(root):
            return False
        children = [
            name for name in os.listdir(root)
            if os.path.isdir(os.path.join(root, name))
        ]
        class_like = [
            name for name in children
            if name not in ('train', 'val', 'test', '__MACOSX')
        ]
        return (
            len(class_like) >= 40
            and not {'train', 'val', 'test'}.issubset(set(children))
        )

    raw_root = os.environ.get('RAW_DATASET_ROOT', '').strip()
    if not raw_root:
        for candidate in sorted(os.listdir('/kaggle/input')):
            path = os.path.join('/kaggle/input', candidate)
            if _looks_like_raw_class_tree(path):
                raw_root = path
                break

    if raw_root:
        print(f'Raw class-folder source detected: {raw_root}')
        print(f'MAX_PER_CLASS: {max_per_class}')
    else:
        raise RuntimeError(
            f'A rebuild was requested (FORCE_BENCHMARK_REBUILD={force_rebuild}) '
            f'but no raw class-folder dataset was found under /kaggle/input. '
            'Attach the uncapped raw dataset (dataset/raw) via Add Input, or '
            'set the RAW_DATASET_ROOT environment variable. Rebuilding from '
            'the mounted train/val/test tree cannot raise the per-class cap.'
        )

    if os.path.isfile(benchmark_manifest):
        print('Deduplicated benchmark already present:', benchmark_dir)
        print('Delete that directory first to rebuild with a different cap.')
    else:
        if os.path.exists(benchmark_dir):
            raise RuntimeError(
                'Incomplete benchmark at {} - delete only that directory and '
                'rerun this cell.'.format(benchmark_dir)
            )
        if repo_dir not in sys.path:
            sys.path.insert(0, repo_dir)
        from dataset.prepare_benchmark import prepare_benchmark

        result = prepare_benchmark(
            raw_root,
            benchmark_dir,
            seed=42,
            max_per_class=max_per_class,
        )
        print('Prepared deduplicated benchmark:', result['counts'])
        print('Degenerate classes (disclosed val/test copies):',
              result['degenerate_classes'])
        print('Now re-run the configure cell above to switch to this benchmark.')


## Train the model

In [ ]:
import os
import subprocess
import sys
import time
import yaml
training_dir = os.path.join(REPO_DIR, 'training')
if training_dir not in sys.path:
    sys.path.insert(0, training_dir)

from runtime_profiles import apply_runtime_profile
os.chdir(REPO_DIR)

training_dir = os.path.join(REPO_DIR, 'training')

RUN_MODE = os.environ.get('TRAINING_RUN_MODE', 'full').strip().lower()
DIAGNOSTIC = os.environ.get('TRAINING_DIAGNOSTIC', '').strip().lower() in {'1', 'true', 'yes', 'on'}

config_path = runtime_cfg_path  # reuse the Kaggle-modified copy from the previous cell
with open(config_path, encoding='utf-8') as config_file:
    training_config = yaml.safe_load(config_file)
training_config = apply_runtime_profile(training_config, RUN_MODE, platform='kaggle', diagnostic=DIAGNOSTIC)
# Hosted streams render "\r" in place, so training repaints one
# live status line instead of appending a line per batch.
training_config['training']['progress_style'] = 'inline'
runtime_config_name = 'config_mobilenetv2_kaggle_diagnostic.yaml' if DIAGNOSTIC else 'config_mobilenetv2_kaggle_safe.yaml'
runtime_config_path = os.path.join(training_dir, runtime_config_name)
with open(runtime_config_path, 'w', encoding='utf-8') as config_file:
    yaml.safe_dump(training_config, config_file, sort_keys=False)
print(f'Training run mode: {RUN_MODE}; diagnostic={DIAGNOSTIC}')
print('Kaggle runtime settings:', f"mode={RUN_MODE}, batch_size={training_config['dataset']['batch_size']}, distribution={training_config['performance']['distribution']}, multi_gpu={training_config['performance'].get('multi_gpu', False)}, cache_mode={training_config['dataset'].get('cache_mode', 'none')}, steps_per_execution={training_config['performance']['steps_per_execution']}, jit_compile={training_config['performance']['jit_compile']}")

training_command = [
    sys.executable, '-X', 'faulthandler', '-u',
    os.path.join(training_dir, 'train.py'),
    '--config', runtime_config_path,
]
runtime_environment = dict(os.environ)
runtime_environment['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
runtime_environment['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Starting training subprocess; child output will stream below...', flush=True)
training_started_at = time.perf_counter()
training_process = subprocess.Popen(
    training_command,
    cwd=training_dir,
    env=runtime_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
training_output = []
assert training_process.stdout is not None
# Chunked reads, not a line iterator: train.py repaints one live
# status line with "\r", and a line iterator would buffer every
# repaint until the end-of-epoch newline.
while True:
    output_chunk = training_process.stdout.read(4096)
    if not output_chunk:
        break
    output_text = output_chunk.decode('utf-8', errors='replace')
    print(output_text, end='', flush=True)
    training_output.append(output_text)
training_process.stdout.close()
training_returncode = training_process.wait()
training_diagnostics = ''.join(training_output)
print(f"Training subprocess runtime: {time.perf_counter() - training_started_at:.1f}s")
if training_returncode != 0:
    raise RuntimeError(
        'Training process failed with exit code '
        f'{training_returncode}. Child-process output:\n{training_diagnostics}'
    )

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(ARCH_OUTPUT_DIR)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(ARCH_OUTPUT_DIR, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", runtime_config_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
eval_script = os.path.join(REPO_DIR, "quantization", "evaluate_tflite.py")
quant_dir = os.path.join(ARCH_OUTPUT_DIR, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--arch", ARCH,
], check=True)

# Full-test-set metrics for both quantized variants (feeds summarize_runs.py).
for variant, artifact in (("dynamic", dynamic_path), ("int8", int8_path)):
    subprocess.run([
        sys.executable, eval_script,
        "--model", artifact,
        "--test-data", cfg["dataset"]["test_dir"],
        "--output-dir", os.path.join(ARCH_OUTPUT_DIR, f"tflite_metrics_{variant}"),
        "--arch", ARCH,
    ], check=True)

print("Exported:", dynamic_path, int8_path)


## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(ARCH_OUTPUT_DIR, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", ARCH_OUTPUT_DIR)
print("Archive:", archive_path)

In [ ]:
# Download link: the Output panel button silently refuses files over
# ~20 MB, and the exported archive is larger than that. FileLink serves
# the zip from this live session, so the browser can download it.
# FileLink resolves both the existence check and the rendered link against
# the kernel working directory, and the training cells left it inside the
# cloned repo -- move back to the file root first, or the display degrades
# into a "Path (...) doesn't exist" notice.
# The link works only while the interactive session is running; for a
# committed version pull the output with the Kaggle API instead.
import os
from IPython.display import FileLink, display

os.chdir(os.path.dirname(archive_path))

print(f"Archive size: {os.path.getsize(archive_path) / 1e6:.1f} MB")
display(FileLink(os.path.basename(archive_path)))  # click to download the zip

# Belt and braces: the panel also refuses every single file over ~20 MB, so
# split the archive into ~19 MB parts it will serve. Rejoin them afterwards
# with: copy /b <stem>.part00+<stem>.part01+... <name>.zip   (Windows)
#       cat <stem>.part* > <name>.zip                       (macOS/Linux)
MAX_PART_BYTES = 19 * 1024 * 1024
archive_name = os.path.basename(archive_path)
if os.path.getsize(archive_path) > MAX_PART_BYTES:
    stem = archive_name[: -len(".zip")]
    prefix = stem + ".part"
    for entry in os.listdir("."):
        if entry.startswith(prefix) and entry[len(prefix) :].isdecimal():
            os.remove(entry)
    with open(archive_path, "rb") as payload:
        part_index = 0
        while True:
            chunk = payload.read(MAX_PART_BYTES)
            if not chunk:
                break
            part_name = f"{prefix}{part_index:02d}"
            with open(part_name, "wb") as part_file:
                part_file.write(chunk)
            print(f"{part_name}: {len(chunk) / 1e6:.1f} MB")
            display(FileLink(part_name))
            part_index += 1
    parts = sorted(
        entry
        for entry in os.listdir(".")
        if entry.startswith(prefix) and entry[len(prefix) :].isdecimal()
    )
    print("Rejoin with: copy /b " + "+".join(parts) + f" {archive_name}  (Windows)")
    print(f"or: cat {prefix}* > {archive_name}  (macOS/Linux)")


## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.

**Downloading the archive:** Kaggle's Output panel only handles files up to
~20 MB, so the archive above cannot be pulled from that button. The cell above
renders a `FileLink` that streams the zip directly from the live session - click
it while the session is still running (`Save & Run All` versions and stopped
sessions cannot serve it). For committed versions, pull the output with the
Kaggle API instead:

```bash
kaggle kernels output <your-username>/<notebook-slug> -p ./kaggle_out
```

If the archive is larger than ~20 MB, the cell also writes `<stem>.part00`-style splits that the panel will serve; rejoin them with `copy /b <stem>.part00+... <name>.zip` (Windows) or `cat <stem>.part* > <name>.zip` (macOS/Linux).
